# Lane B #9 elastic-net leg - Colab offload

Runs the seeded enet pass 2 of `scripts/p2_multitask.py` (random_state=20260929, commit 0a5e1df).

Run order: Cell 1 (data), Cell 2 (pinned install, auto-restarts runtime), then after reconnect Cell 3 (replay check - MUST print REPLAY PASS), Cell 4 (enet run), Cell 5 (final verification).

Data: 4 release assets (offload-data-v1{,-ab,-ac,-ad}) -> cat -> md5 226cf5f54f838b6582b8a80cde9b898f -> extract into repo root. No Drive mount required: fold results stream to this cell's output as they complete (per-fold checkpoint lines), and the full fold records dump at the end between ENET_FOLD_RECORDS_BEGIN/END markers. After a session drop: re-run Cells 1,3,4 - the script skips folds already in the local JSON; fold lines already printed below remain the durable record.


In [ ]:
import os, subprocess, hashlib
os.chdir('/content')
if not os.path.isdir('mega27-17s-spatial-morphoscan'):
    subprocess.run(['git','clone','--depth','1','https://github.com/uditakankananonononono/mega27-17s-spatial-morphoscan.git'], check=True)
REPO='/content/mega27-17s-spatial-morphoscan'
base='https://github.com/uditakankananonononono/mega27-17s-spatial-morphoscan/releases/download'
for tag,part in [('offload-data-v1','part_aa'),('offload-data-v1-ab','part_ab'),('offload-data-v1-ac','part_ac'),('offload-data-v1-ad','part_ad')]:
    fn=f'laneB_her2_data.{part}'
    subprocess.run(['curl','-sL','-o',f'/content/{fn}',f'{base}/{tag}/{fn}'], check=True)
subprocess.run('cat /content/laneB_her2_data.part_a* > /content/laneB_her2_data.tar.gz', shell=True, check=True)
h=hashlib.md5(open('/content/laneB_her2_data.tar.gz','rb').read()).hexdigest()
assert h=='226cf5f54f838b6582b8a80cde9b898f', f'MD5 MISMATCH {h}'
print('MD5 OK', h)
subprocess.run(['tar','-xzf','/content/laneB_her2_data.tar.gz','-C',REPO], check=True)
print('EXTRACTED into', REPO)


In [ ]:
%pip install -q -r /content/mega27-17s-spatial-morphoscan/requirements_offload.txt
import os
os.kill(os.getpid(), 9)  # force runtime restart so pinned numpy/scipy/sklearn load cleanly


In [ ]:
import os
REPO='/content/mega27-17s-spatial-morphoscan'
os.environ['MS_DATA']=REPO+'/data/her2st'
os.chdir(REPO)
import numpy, scipy, sklearn, pandas
print('numpy', numpy.__version__, '| scipy', scipy.__version__, '| sklearn', sklearn.__version__, '| pandas', pandas.__version__)
assert numpy.__version__=='2.2.6' and sklearn.__version__=='1.7.2', 'pinned versions not loaded - restart runtime and re-run this cell'
!python scripts/offload_replay_check.py


In [ ]:
import os, subprocess, time, json, re
REPO='/content/mega27-17s-spatial-morphoscan'
os.environ['MS_DATA']=REPO+'/data/her2st'
os.chdir(REPO)
p=subprocess.Popen(['python','-u','scripts/p2_multitask.py'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=os.environ)
t0=time.time()
logf=open('/content/enet_run.log','w')
for line in p.stdout:
    logf.write(line); logf.flush()
    print(line, end='', flush=True)
p.wait()
print(f'PROCESS EXITED rc={p.returncode} total {(time.time()-t0)/60:.1f} min')
d=json.load(open(REPO+'/results/p2_multitask.json'))
enet=[f for f in d.get('folds',[]) if 'elasticnet_median_r' in f]
print('ENET_FOLD_RECORDS_BEGIN')
for f in enet:
    print(json.dumps({'test_patient': f['test_patient'], 'elasticnet_median_r': f['elasticnet_median_r'], 'elasticnet_n_iter': f.get('elasticnet_n_iter')}))
print('ENET_FOLD_RECORDS_END')


In [ ]:
import json, hashlib
p='/content/mega27-17s-spatial-morphoscan/results/p2_multitask.json'
d=json.load(open(p))
print('json md5:', hashlib.md5(open(p,'rb').read()).hexdigest())
enet=[(f['test_patient'], f['elasticnet_median_r'], f.get('elasticnet_n_iter')) for f in d.get('folds',[]) if 'elasticnet_median_r' in f]
print('enet folds:', len(enet))
for t,r,n in enet: print(f'  {t}: enet_median_r={r:.6f} n_iter={n}')
print('summary:', json.dumps(d.get('enet_summary', d.get('summary', 'see json')), indent=2)[:800])
